In [1]:
import numpy as np
import pandas as pd
import os
from PIL import Image
import pydicom
import matplotlib.pyplot as plt
import pylab
import cv2
from tensorflow.keras.utils import Sequence
#import gdcm
from tqdm import tqdm

import tensorflow as tf
import tensorflow.keras.layers as L
import tensorflow.keras.models as M
import tensorflow.keras.backend as K
import tensorflow.keras.regularizers as R
from sklearn.metrics import mean_absolute_error


2026-01-07 06:27:14.311922: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767767234.325248      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767767234.329221      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-07 06:27:14.345532: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
comp_dir = '../input/osic-pulmonary-fibrosis-progression'

train_data = pd.read_csv(os.path.join(comp_dir,'train.csv'))
sub = pd.read_csv(os.path.join(comp_dir,'sample_submission.csv'))
test_data = pd.read_csv(os.path.join(comp_dir,'test.csv'))
train_data.drop_duplicates(keep=False, inplace=True, subset=['Patient','Weeks'])


In [3]:
train_data.head()


,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus
0,ID00133637202223847701934,-2,3195,92.856312,83,Male,Never smoked
1,ID00133637202223847701934,2,3203,93.088817,83,Male,Never smoked
2,ID00133637202223847701934,4,3097,90.008138,83,Male,Never smoked
3,ID00133637202223847701934,6,3171,92.158800,83,Male,Never smoked
4,ID00133637202223847701934,8,3115,90.531272,83,Male,Never smoked


In [4]:
test_data.head()


,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus
0,ID00014637202177757139317,0,3807,90.076661,56,Male,Ex-smoker
1,ID00019637202178323708467,13,2100,92.858722,83,Female,Ex-smoker
2,ID00047637202184938901501,2,3313,89.929425,68,Male,Ex-smoker
3,ID00082637202201836229724,19,2918,99.794802,49,Female,Currently smokes
4,ID00126637202218610655908,18,2375,59.375000,78,Male,Ex-smoker


In [5]:
sub.head()


,Patient_Week,FVC,Confidence
0,ID00126637202218610655908_-3,2000,100
1,ID00126637202218610655908_-2,2000,100
2,ID00126637202218610655908_-1,2000,100
3,ID00126637202218610655908_0,2000,100
4,ID00126637202218610655908_1,2000,100


In [6]:
train_data_u = train_data
train_data_u = train_data_u.drop_duplicates(subset=['Patient'])
train_data_u = train_data_u.rename(columns={'Weeks':'Base_Week','FVC':'Base_FVC','Percent':'Base_Percent'})
train_data_u['Typical_FVC'] = (train_data_u.Base_FVC.values/train_data_u.Base_Percent.values)*100
train_data = train_data.merge(train_data_u.drop(['Age','Sex','SmokingStatus'],axis=1),on='Patient',how='left')


In [7]:
sub['Patient'] = sub['Patient_Week'].apply(lambda x:x.split('_')[0])
sub['Weeks'] = sub['Patient_Week'].apply(lambda x: int(x.split('_')[-1]))
sub = sub.drop(['Confidence'],axis=1)
sub =  sub[['Patient','Weeks','Patient_Week']]


In [8]:
test_data = test_data.rename(columns={'Weeks':'Base_Week','FVC':'Base_FVC','Percent':'Base_Percent'})
test_data['Typical_FVC'] = (test_data.Base_FVC.values/test_data.Base_Percent.values)*100
sub = sub.merge(test_data, how='left', on='Patient')


In [9]:
train_data['Type'] = 'train'
sub['Type'] = 'test'


In [10]:
data = train_data.append(sub)


AttributeError: 'DataFrame' object has no attribute 'append'

In [11]:
prediction_col = ["FVC"]
Continuos_cols = ["Weeks","Base_Week","Base_FVC","Typical_FVC","Age","Percent","Base_Percent"]
Categorical_cols = ['Sex','Smoking_status']


In [12]:
from sklearn.preprocessing import MinMaxScaler


In [13]:
scaler = MinMaxScaler()
conti = scaler.fit_transform(data[Continuos_cols])
data[Continuos_cols] = conti


NameError: name 'data' is not defined

In [14]:
print(np.mean(train_data_u.query('SmokingStatus == \'Never smoked\'').Base_Percent.values))
print(np.mean(train_data_u.query('SmokingStatus == \'Currently smokes\'').Base_Percent.values))
print(np.mean(train_data_u.query('SmokingStatus == \'Ex-smoker\'').Base_Percent.values))
print(np.mean(train_data_u.query('Sex == \'Male\'').Base_Percent.values))
print(np.mean(train_data_u.query('Sex == \'Female\'').Base_Percent.values))


78.68346533571169
100.4693934999359
78.08689959858515
77.95708401998029
84.33313954809648


In [15]:
for i in range(len(data['Sex'].values)):
    if data['Sex'].values[i] =='Male':
        data['Sex'].values[i] = 0
    else:
        data['Sex'].values[i] = 1

for i in range(len(data['SmokingStatus'].values)):
    if data['SmokingStatus'].values[i] =='Ex-smoker':
        data['SmokingStatus'].values[i] = 0
    elif data['SmokingStatus'].values[i] =='Never smoked':
        data['SmokingStatus'].values[i] = 1     
    else:
        data['SmokingStatus'].values[i] = 2 


NameError: name 'data' is not defined

In [16]:
data


NameError: name 'data' is not defined

In [17]:
x_cols = ['Weeks','Base_Week','Sex','SmokingStatus','Age','Base_Percent','Typical_FVC']


In [18]:
data


NameError: name 'data' is not defined

In [19]:
x_train = data[x_cols].loc[data['Type'] == "train"].values
y_train = data[prediction_col].loc[data['Type'] == "train"].values
x_test = data[x_cols].loc[data['Type'] == "test"].values


NameError: name 'data' is not defined

In [20]:
x_train = x_train.astype(np.float32)
y_train = y_train.astype(np.float32)
x_test = x_test.astype(np.float32)


NameError: name 'x_train' is not defined

In [21]:
x_train.shape,y_train.shape,x_test.shape


NameError: name 'x_train' is not defined

In [22]:
type(x_train),type(y_train),type(x_test)


NameError: name 'x_train' is not defined

In [23]:
C1, C2 = tf.constant(70, dtype='float32'), tf.constant(1000, dtype="float32")
#=============================#
def score(y_true, y_pred):
    tf.dtypes.cast(y_true, tf.float32)
    tf.dtypes.cast(y_pred, tf.float32)
    sigma = y_pred[:,2]-y_pred[:,0]
    fvc_pred = y_pred[:,1]
    #sigma_clip = sigma + C1
    sigma_clip = tf.maximum(sigma, C1)
    delta = tf.abs(y_true[:,0] - fvc_pred)
    delta = tf.minimum(delta, C2)
    sq2 = tf.sqrt( tf.dtypes.cast(2, dtype=tf.float32) )
    metric = (delta / sigma_clip)*sq2 + tf.math.log(sigma_clip* sq2)
    return K.mean(metric)
#============================#
def qloss(y_true, y_pred):
    # Pinball loss for multiple quantiles
    qs = [0.2,0.50,0.8]
    q = tf.constant(np.array([qs]), dtype=tf.float32)
    e = y_true - y_pred
    v = tf.maximum(q*e,(q-1)*e)
    return K.mean(v)
#=============================#
def mloss(_lambda):
    def loss(y_true, y_pred):
        return _lambda * qloss(y_true, y_pred) + (1 - _lambda)*score(y_true, y_pred)
    return loss
#=================


2026-01-07 06:27:21.475514: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1767767241.475976      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:85:00.0, compute capability: 8.6


In [24]:
def build_model():
    
    inp = L.Input((7,))
    x = L.Dense(128,activation ='relu',kernel_regularizer=R.l2(1e-6))(inp)
    x = L.Dense(128,activation ='relu',kernel_regularizer=R.l2(1e-6))(x)
    x = L.Dense(64,activation ='relu',kernel_regularizer=R.l2(1e-6))(x)
    o1 = L.Dense(3,activation = 'linear')(x)
    o2 = L.Dense(3,activation = 'relu')(x)
    pred1 = L.Lambda(lambda x: (x[0]+(tf.cumsum(x[1],axis=1))))([o1,o2])    
    
    
    model = M.Model(inputs=inp,outputs=pred1)
    
    model.compile(loss=mloss(0.8), optimizer=tf.keras.optimizers.Adam(learning_rate=1e-2), metrics=[score])
    
    return model
     


In [25]:
model = build_model()
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │      1,024 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     16,512 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 3)         │        195 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 3)         │        195 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 3)         │          0 │ dense_3[0][0],    │
│                     │                   │            │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 26,182 (102.27 KB)

 Trainable params: 26,182 (102.27 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
from sklearn.model_selection import KFold
folds = 5
KF = KFold(n_splits=folds,shuffle=True,random_state=24)


In [27]:
%%time
counter=0
stopper = tf.keras.callbacks.EarlyStopping(monitor='loss',mode='min',patience=200,restore_best_weights=True)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.1,patience=50, min_lr=1e-3)
for tr_idx, val_idx in KF.split(x_train):
    counter+=1
    print(f'############## FOLD {counter} ###############')
    model.fit(x_train[tr_idx],y_train[tr_idx],epochs=800,batch_size = 256,verbose=0,validation_data=(x_train[val_idx],y_train[val_idx]),callbacks=[reduce_lr])
    print("train", model.evaluate(x_train[tr_idx], y_train[tr_idx], verbose=0, batch_size=256))
    print("val", model.evaluate(x_train[val_idx], y_train[val_idx], verbose=0, batch_size=256))


NameError: name 'x_train' is not defined

In [28]:
model.save('model.h5')


In [29]:
pred = model.predict(x_train,verbose=1)


NameError: name 'x_train' is not defined

In [30]:
sigma_opt = mean_absolute_error(y_train, pred[:,1])
unc = pred[:,2] - pred[:, 0]
sigma_mean = np.mean(unc)
sigma = pred[:,2]-pred[:,0]
print(sigma_opt,sigma_mean,sigma)


NameError: name 'y_train' is not defined

In [31]:
pred[:10]


NameError: name 'pred' is not defined

In [32]:
pred = model.predict(x_test,verbose=1)


NameError: name 'x_test' is not defined

In [33]:
pred


NameError: name 'pred' is not defined

In [34]:
conf = pred[:,2] - pred[:,0]
for i in range(len(conf)):
    conf[i] = max(conf[i],70)


NameError: name 'pred' is not defined

In [35]:
pred_dict = {'FVC':pred[:,1],'Confidence':conf}
pred_df = pd.DataFrame(pred_dict)


NameError: name 'pred' is not defined

In [36]:
sub['Confidence'] = pred_df['Confidence']
sub['FVC'] = pred_df['FVC']


NameError: name 'pred_df' is not defined

In [37]:
subm = sub[['Patient_Week','FVC','Confidence']].copy()


KeyError: "['FVC', 'Confidence'] not in index"

In [38]:
subm[300:400]


NameError: name 'subm' is not defined

In [39]:
subm.to_csv("submission.csv", index=False)


NameError: name 'subm' is not defined